# GraphRAG with a Milvus Vector DB

Store relations as vectors in Milvus, retrieve the closest, then expand through the graph.

**Extra requirements:** `pip install pymilvus networkx`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx
from pymilvus import MilvusClient

G = nx.Graph()
relations = []
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r)
        relations.append(f"{s} {r} {o}")
print(len(relations), "relations")

Milvus Lite runs from a local file - no server required.

In [ ]:
client = MilvusClient(str(DATA / "graph.db"))
if client.has_collection("relations"):
    client.drop_collection("relations")
client.create_collection(collection_name="relations", dimension=embed("x").shape[1])

vecs = embed(relations)
client.insert("relations", [{"id": i, "vector": vecs[i].tolist(), "text": relations[i]} for i in range(len(relations))])

In [ ]:
def milvus_graph_search(query, k=5, hops=1):
    res = client.search("relations", data=[embed(query)[0].tolist()], limit=k, output_fields=["text"])[0]
    seeds = [r["entity"]["text"] for r in res]
    # expand: add other edges touching the entities of the seed relations
    entities = {n for n in G.nodes if any(n in s for s in seeds)}
    expanded = set(seeds)
    for n in entities:
        for nb in G.neighbors(n):
            expanded.add(f"{n} {G[n][nb]['relation']} {nb}")
    return seeds, sorted(expanded)[:25]

query = "How do vector databases speed up nearest neighbour search?"
seeds, context = milvus_graph_search(query)
print("seed relations:", seeds)
print(answer(query, context))